In [ ]:
from ultralytics import YOLO

model = YOLO("yolov8l.pt")

model.train(
    data=r"D:\DMSc_Dissertation_Project\datasets\DAWN\YOLO_ENTIRE\dawn_entire.yaml",
    epochs=100,
    imgsz=640,
    batch=2,
    workers=2,
    device=0,
    cache=False,
    amp=True,
    project="DAWN_Entire_Project",
    name="train_yolov8l_entire"
)

In [ ]:
import os
import shutil
import pandas as pd

# 1. Configuration
PROJECT_DIR = r"C:\Users\Varis\runs\detect\DAWN_Entire_Project\train_yolov8l_entire-3"
BACKUP_DIR = r"D:\DMSc_Dissertation_Project\datasets\DAWN\YOLO_ENTIRE"

# Ensure backup directory exists
os.makedirs(BACKUP_DIR, exist_ok=True)

# 2. Define source paths
weights_dir = os.path.join(PROJECT_DIR, "weights")

# List of all files to move
files_to_copy = [
    "args.yaml", "results.csv", "results.png", "confusion_matrix.png", 
    "confusion_matrix_normalized.png", "labels.jpg"
]

# Specifically look for curve files (YOLOv8 sometimes prefixes them with 'Box')
curve_files = [f for f in os.listdir(PROJECT_DIR) if "_curve.png" in f or "batch" in f]

print("Starting the backup process...")

# Copy standard files
for file in files_to_copy:
    if os.path.exists(os.path.join(PROJECT_DIR, file)):
        shutil.copy2(os.path.join(PROJECT_DIR, file), BACKUP_DIR)
        print(f"Successfully archived: {file}")

# Copy curves and batches
for file in curve_files:
    shutil.copy2(os.path.join(PROJECT_DIR, file), BACKUP_DIR)
    print(f"Successfully archived: {file}")

# Copy weights
for weight_file in ["best.pt", "last.pt"]:
    if os.path.exists(os.path.join(weights_dir, weight_file)):
        shutil.copy2(os.path.join(weights_dir, weight_file), BACKUP_DIR)
        print(f"Successfully archived: {weight_file}")

# 3. Create Performance Summary for Dissertation
results_df = pd.read_csv(os.path.join(PROJECT_DIR, "results.csv"))
results_df.columns = results_df.columns.str.strip()
summary_df = pd.DataFrame([{
    "Project": "DAWN_ENTIRE",
    "Max_mAP50": results_df['metrics/mAP50(B)'].max(),
    "Max_mAP50-95": results_df['metrics/mAP50-95(B)'].max()
}])
summary_df.to_csv(os.path.join(BACKUP_DIR, "performance_summary.csv"), index=False)

print(f"\nBackup complete! All files moved to: {BACKUP_DIR}")